# Parallel Self-Play

Notebook này chạy `self_play_parallel.py` từ repo root và lưu dữ liệu self-play `.pt` cho racetrack.

In [ ]:
from pathlib import Path
import shlex
import subprocess


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "AlphaZero-based-autonomous-driving").is_dir() and (candidate / "highway-env").is_dir():
            return candidate
    raise RuntimeError("Could not locate repo root. Open the notebook from inside this repository.")


def run_command(command, cwd: Path):
    print("$ " + " ".join(shlex.quote(str(part)) for part in command))
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}")


repo_root = find_repo_root()
package_root = repo_root / "AlphaZero-based-autonomous-driving"
runner = ["uv", "run", "python", "-m"]
self_play_module = "AlphaZero.scripts.self_play_parallel"
print(f"repo_root={repo_root}")
print(f"package_root={package_root}")
print(f"self_play_module={self_play_module}")

In [ ]:
output_dir = repo_root / "AlphaZero-based-autonomous-driving" / "outputs" / "self_play_parallel"

config = {
    "workers": 2,
    "episodes_per_worker": 2,
    "duration": 80,
    "other_vehicles": 1,
    "finish_laps": 1,
    "terminate_on_finish": True,
    "n_simulations": 5,
    "c_puct": 2.5,
    "temperature": 1.0,
    "temperature_drop_step": None,
    "dirichlet_alpha": 0.3,
    "root_exploration_fraction": 0.25,
    "n_residual_layers": 10,
    "device": "auto",
    "torch_threads_per_worker": 1,
    "progress_interval": 5,
    "max_steps_per_episode": None,
    "model_path": None,
    "output_dir": output_dir,
}

config

In [ ]:
command = [
    *runner,
    self_play_module,
    "--workers", str(config["workers"]),
    "--episodes-per-worker", str(config["episodes_per_worker"]),
    "--duration", str(config["duration"]),
    "--other-vehicles", str(config["other_vehicles"]),
    "--finish-laps", str(config["finish_laps"]),
    "--n-simulations", str(config["n_simulations"]),
    "--c-puct", str(config["c_puct"]),
    "--temperature", str(config["temperature"]),
    "--n-residual-layers", str(config["n_residual_layers"]),
    "--device", str(config["device"]),
    "--torch-threads-per-worker", str(config["torch_threads_per_worker"]),
    "--progress-interval", str(config["progress_interval"]),
    "--output-dir", str(config["output_dir"]),
]

command.append("--terminate-on-finish" if config["terminate_on_finish"] else "--no-terminate-on-finish")

if config["temperature_drop_step"] is not None:
    command.extend(["--temperature-drop-step", str(config["temperature_drop_step"])])
if config["dirichlet_alpha"] is not None:
    command.extend(["--dirichlet-alpha", str(config["dirichlet_alpha"])])
if config["root_exploration_fraction"] is not None:
    command.extend(["--root-exploration-fraction", str(config["root_exploration_fraction"])])
if config["max_steps_per_episode"] is not None:
    command.extend(["--max-steps-per-episode", str(config["max_steps_per_episode"])])
if config["model_path"] is not None:
    command.extend(["--model-path", str(config["model_path"])])

run_command(command, cwd=package_root)